# Deep Learning Models for Oil Recovery Factor Prediction
## Comparative Study: MLP · LSTM · CNN · Transformer

**Dataset:** Proxy5 — polymer flood reservoir simulation  
**Target:** `Oil_recovery_factor (%)`  
**Features:** 14 reservoir / fluid / operational parameters

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Hyper-parameters (shared) ────────────────────────────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 200
LR           = 1e-3
PATIENCE     = 20      # early-stopping patience
TEST_SIZE    = 0.15
VAL_SIZE     = 0.15    # fraction of remaining train data used for validation

---
## 2. Data Loading & Exploration

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
# Place Proxy5.csv in the same folder as this notebook, or adjust the path.
DATA_PATH = 'Proxy5.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
TARGET  = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Features ({len(FEATURES)}):', FEATURES)

In [ ]:
# Distribution of target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Oil Recovery Factor — raw distribution')
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

axes[1].boxplot(df[TARGET], vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Box plot')
axes[1].set_xlabel(TARGET)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Pre-processing

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32).reshape(-1, 1)

# Train / val / test split  (70 / 15 / 15)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# Scale features and target
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s   = x_scaler.transform(X_val)
X_test_s  = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s   = y_scaler.transform(y_val)
y_test_s  = y_scaler.transform(y_test)

# ── PyTorch tensors ───────────────────────────────────────────────────────────
def to_tensor(*arrays):
    return [torch.tensor(a, dtype=torch.float32).to(DEVICE) for a in arrays]

Xt, yt         = to_tensor(X_train_s, y_train_s)
Xv, yv         = to_tensor(X_val_s,   y_val_s)
Xte, yte       = to_tensor(X_test_s,  y_test_s)

train_loader = DataLoader(TensorDataset(Xt, yt),   batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=BATCH_SIZE)

N_FEATURES = X_train_s.shape[1]
print(f'Input dimension: {N_FEATURES}')

---
## 4. Training Utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS,
                lr=LR, patience=PATIENCE, model_name='model'):
    """Generic training loop with early stopping."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=False)

    best_val_loss = np.inf
    best_state    = None
    no_improve    = 0
    history       = {'train': [], 'val': []}

    for epoch in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        # ── validate ─────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                val_loss += criterion(pred, yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1

        if epoch % 20 == 0:
            print(f'[{model_name}] Epoch {epoch:>3}/{epochs}  '
                  f'train={train_loss:.5f}  val={val_loss:.5f}')

        if no_improve >= patience:
            print(f'[{model_name}] Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, history


def evaluate(model, loader, y_scaler):
    """Return inverse-scaled predictions and ground truth plus metrics."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    # Inverse-scale
    preds_inv = y_scaler.inverse_transform(preds)
    trues_inv = y_scaler.inverse_transform(trues)
    rmse  = np.sqrt(mean_squared_error(trues_inv, preds_inv))
    mae   = mean_absolute_error(trues_inv, preds_inv)
    r2    = r2_score(trues_inv, preds_inv)
    mape  = np.mean(np.abs((trues_inv - preds_inv) /
                           np.where(trues_inv == 0, 1e-8, trues_inv))) * 100
    return preds_inv, trues_inv, {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

---
## 5. Model 1 — Multi-Layer Perceptron (MLP)

A fully-connected feed-forward network.  
It is the most common baseline DL regressor and often competitive on tabular data.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


mlp_model = MLP(N_FEATURES).to(DEVICE)
print(mlp_model)
n_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

In [ ]:
mlp_model, mlp_history = train_model(
    mlp_model, train_loader, val_loader, model_name='MLP')

mlp_preds, mlp_trues, mlp_metrics = evaluate(mlp_model, test_loader, y_scaler)
print('\nMLP Test metrics:', mlp_metrics)

---
## 6. Model 2 — 1-D Convolutional Neural Network (CNN)

1-D convolutions treat the 14 input features as a sequence and learn local feature interactions.  
CNNs capture spatial/local correlations and are computationally efficient.

In [ ]:
class CNN1D(nn.Module):
    """
    Treats the feature vector as a 1-D sequence (channels=1).
    Three Conv1d blocks → global average pooling → FC head.
    """
    def __init__(self, in_dim, dropout=0.3):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            # Block 1
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 2
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        # Global average pooling over the feature dimension
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (batch, features)  →  (batch, 1, features)
        x = x.unsqueeze(1)
        x = self.conv_blocks(x)   # (batch, 128, features)
        x = self.gap(x)           # (batch, 128, 1)
        return self.head(x)       # (batch, 1)


cnn_model = CNN1D(N_FEATURES).to(DEVICE)
print(cnn_model)
n_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

In [ ]:
cnn_model, cnn_history = train_model(
    cnn_model, train_loader, val_loader, model_name='CNN')

cnn_preds, cnn_trues, cnn_metrics = evaluate(cnn_model, test_loader, y_scaler)
print('\nCNN Test metrics:', cnn_metrics)

---
## 7. Model 3 — LSTM (Long Short-Term Memory)

Although the data is tabular (not temporal), treating features as a sequence lets the LSTM learn
ordered feature interactions via its gating mechanism.  
LSTMs also generalise well when features have natural ordering (e.g., by importance or physical grouping).

In [ ]:
class LSTMRegressor(nn.Module):
    """
    Each feature is a time-step of a univariate signal (input_size=1).
    Two stacked LSTM layers → last hidden state → FC head.
    """
    def __init__(self, in_dim, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (batch, features)  →  (batch, features, 1)
        x = x.unsqueeze(-1)
        out, (h_n, _) = self.lstm(x)   # h_n: (num_layers, batch, hidden)
        # Use the last layer's hidden state
        h_last = self.dropout(h_n[-1])  # (batch, hidden)
        return self.head(h_last)        # (batch, 1)


lstm_model = LSTMRegressor(N_FEATURES).to(DEVICE)
print(lstm_model)
n_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

In [ ]:
lstm_model, lstm_history = train_model(
    lstm_model, train_loader, val_loader, model_name='LSTM')

lstm_preds, lstm_trues, lstm_metrics = evaluate(lstm_model, test_loader, y_scaler)
print('\nLSTM Test metrics:', lstm_metrics)

---
## 8. Model 4 — Transformer (Tabular Transformer)

The Transformer encoder uses self-attention to weight pairwise feature relationships,
making it well-suited for tabular data with complex inter-feature dependencies.  
Each feature is embedded into a *d_model*-dimensional space before multi-head attention.

In [ ]:
class TabularTransformer(nn.Module):
    """
    Embeds each scalar feature into d_model dimensions, then applies
    nhead-head self-attention across features.
    The [CLS] token aggregates information → FC regression head.
    """
    def __init__(self, in_dim, d_model=64, nhead=4,
                 num_encoder_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        # Per-feature linear embedding
        self.feature_embed = nn.Linear(1, d_model)
        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        # Positional encoding (learnable)
        self.pos_embed = nn.Parameter(torch.zeros(1, in_dim + 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers)

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)
        # x: (B, F)  →  (B, F, 1)  →  (B, F, d_model)
        x = self.feature_embed(x.unsqueeze(-1))
        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)        # (B, F+1, d_model)
        x   = x + self.pos_embed
        x   = self.transformer(x)               # (B, F+1, d_model)
        x   = self.norm(x[:, 0])                # CLS output: (B, d_model)
        return self.head(x)                     # (B, 1)


tfm_model = TabularTransformer(N_FEATURES).to(DEVICE)
print(tfm_model)
n_params = sum(p.numel() for p in tfm_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

In [ ]:
tfm_model, tfm_history = train_model(
    tfm_model, train_loader, val_loader, model_name='Transformer')

tfm_preds, tfm_trues, tfm_metrics = evaluate(tfm_model, test_loader, y_scaler)
print('\nTransformer Test metrics:', tfm_metrics)

---
## 9. Comparative Results

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['MLP', 'CNN-1D', 'LSTM', 'Transformer'],
    'RMSE':  [mlp_metrics['RMSE'],  cnn_metrics['RMSE'],
              lstm_metrics['RMSE'], tfm_metrics['RMSE']],
    'MAE':   [mlp_metrics['MAE'],   cnn_metrics['MAE'],
              lstm_metrics['MAE'],  tfm_metrics['MAE']],
    'R²':    [mlp_metrics['R2'],    cnn_metrics['R2'],
              lstm_metrics['R2'],   tfm_metrics['R2']],
    'MAPE (%)': [mlp_metrics['MAPE'], cnn_metrics['MAPE'],
                 lstm_metrics['MAPE'], tfm_metrics['MAPE']],
})

results = results.sort_values('RMSE').reset_index(drop=True)
results.style.background_gradient(
    subset=['RMSE', 'MAE', 'MAPE (%)'], cmap='RdYlGn_r'
).background_gradient(
    subset=['R²'], cmap='RdYlGn'
).format({
    'RMSE': '{:.4f}', 'MAE': '{:.4f}',
    'R²': '{:.4f}', 'MAPE (%)': '{:.2f}'
})

In [ ]:
# ── Bar charts of metrics ─────────────────────────────────────────────────────
models      = results['Model'].tolist()
colors      = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
color_map   = dict(zip(['MLP', 'CNN-1D', 'LSTM', 'Transformer'], colors))
bar_colors  = [color_map[m] for m in models]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics_to_plot = [('RMSE', True), ('MAE', True), ('R²', False), ('MAPE (%)', True)]
for ax, (metric, lower_better) in zip(axes, metrics_to_plot):
    vals = results[metric].tolist()
    bars = ax.bar(models, vals, color=bar_colors, edgecolor='white', linewidth=1.2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticklabels(models, rotation=15)
    # Annotate bars
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    arrow = '↓ better' if lower_better else '↑ better'
    ax.set_xlabel(arrow, fontsize=10, color='gray')

plt.suptitle('Model Comparison — Oil Recovery Factor Prediction (Test Set)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Training loss curves ──────────────────────────────────────────────────────
histories = {
    'MLP':         mlp_history,
    'CNN-1D':      cnn_history,
    'LSTM':        lstm_history,
    'Transformer': tfm_history,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    c = color_map[name]
    axes[0].plot(hist['train'], label=name, color=c)
    axes[1].plot(hist['val'],   label=name, color=c)

for ax, title in zip(axes, ['Training Loss (MSE)', 'Validation Loss (MSE)']):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (scaled)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Actual vs Predicted scatter plots ────────────────────────────────────────
all_preds = {
    'MLP':         (mlp_trues,  mlp_preds),
    'CNN-1D':      (cnn_trues,  cnn_preds),
    'LSTM':        (lstm_trues, lstm_preds),
    'Transformer': (tfm_trues,  tfm_preds),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.ravel()

for ax, (name, (trues, preds)) in zip(axes, all_preds.items()):
    r2 = r2_score(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    ax.scatter(trues, preds, alpha=0.4, s=15, color=color_map[name], edgecolors='none')
    lo = min(trues.min(), preds.min())
    hi = max(trues.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect fit')
    ax.set_title(f'{name}  (R²={r2:.4f}, RMSE={rmse:.4f})', fontsize=11)
    ax.set_xlabel('Actual Oil Recovery (%)')
    ax.set_ylabel('Predicted Oil Recovery (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.suptitle('Actual vs. Predicted — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, (name, (trues, preds)) in zip(axes, all_preds.items()):
    residuals = trues.ravel() - preds.ravel()
    ax.hist(residuals, bins=50, color=color_map[name], edgecolor='white', alpha=0.8)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name} — Residuals', fontsize=11)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.25)

plt.suptitle('Residual Distributions — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Feature Importance via Permutation

We use **permutation importance**: shuffling one feature at a time and measuring the
increase in test RMSE. Larger increase → more important feature.  
Shown for the best-performing model.

In [ ]:
def permutation_importance(model, X_test_s, y_test, y_scaler,
                            feature_names, n_repeats=5):
    """Return mean and std of RMSE increase for each feature."""
    model.eval()

    # Baseline RMSE
    Xte_t = torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        base_pred = model(Xte_t).cpu().numpy()
    base_pred_inv = y_scaler.inverse_transform(base_pred)
    base_rmse = np.sqrt(mean_squared_error(y_test, base_pred_inv))

    importances = np.zeros((len(feature_names), n_repeats))

    for i, fname in enumerate(feature_names):
        for r in range(n_repeats):
            Xp = X_test_s.copy()
            np.random.shuffle(Xp[:, i])
            Xp_t = torch.tensor(Xp, dtype=torch.float32).to(DEVICE)
            with torch.no_grad():
                pred = model(Xp_t).cpu().numpy()
            pred_inv = y_scaler.inverse_transform(pred)
            perm_rmse = np.sqrt(mean_squared_error(y_test, pred_inv))
            importances[i, r] = perm_rmse - base_rmse

    return importances.mean(axis=1), importances.std(axis=1)


# Pick the best model by RMSE
best_model_name = results.iloc[0]['Model']
best_model_map  = {
    'MLP': mlp_model, 'CNN-1D': cnn_model,
    'LSTM': lstm_model, 'Transformer': tfm_model}
best_model = best_model_map[best_model_name]
print(f'Computing permutation importance for: {best_model_name}')

imp_mean, imp_std = permutation_importance(
    best_model, X_test_s, y_test, y_scaler, FEATURES, n_repeats=5)

sorted_idx = np.argsort(imp_mean)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(
    [FEATURES[i] for i in sorted_idx],
    imp_mean[sorted_idx],
    xerr=imp_std[sorted_idx],
    color='steelblue', alpha=0.8, edgecolor='white'
)
plt.xlabel('Mean RMSE increase (permutation importance)')
plt.title(f'Feature Importance — {best_model_name} (Best Model)', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Final Summary

In [ ]:
print('='*60)
print('   FINAL COMPARISON — OIL RECOVERY FACTOR PREDICTION')
print('='*60)
print(results.to_string(index=False))
print('\nBest model by RMSE:', results.iloc[0]['Model'])
print('Best model by R²  :', results.loc[results['R²'].idxmax(), 'Model'])

---
## Discussion

| Model | Strength | Weakness |
|---|---|---|
| **MLP** | Fast, simple, competitive baseline on tabular data | Does not capture sequential / local correlations |
| **CNN-1D** | Efficient local-feature interaction via convolutions | Feature order is arbitrary; no long-range dependencies |
| **LSTM** | Learns ordered feature interactions; handles varying-importance sequences | Slower to train; less parallelisable |
| **Transformer** | Full pairwise self-attention; most expressive; best on complex interactions | Needs more data to outperform simpler models; higher memory |

**Key takeaway**: For this reservoir simulation dataset the best model is identified above.  
All four models provide physically plausible predictions; the R² > 0.9 threshold indicates strong
predictive power suitable for use as a fast proxy simulator.